In [1]:
import concurrent.futures, requests, json, time, subprocess, sys
from bs4 import BeautifulSoup
import pandas as pd

In [2]:
def extract_ad_info_from_json(ad_data):
    try:
        d, s, p = ad_data.get('detail', {}), ad_data.get('specs', {}), ad_data.get('price', {})
        url_path = d.get('url', '')
        return {
            'ad_code': d.get('code', 'N/A'), 'title': d.get('title', 'N/A'), 'subtitle': d.get('subtitle', 'N/A'),
            'link': f"https://bama.ir{url_path}" if url_path else 'N/A', 'price': p.get('price', 'توافقی'),
            'price_type': p.get('type', 'N/A'), 'location': d.get('location', 'N/A'), 'time_posted': d.get('time', 'N/A'),
            'year': d.get('year', 'N/A'), 'mileage': d.get('mileage', 'N/A'), 'trim': d.get('trim', 'N/A'),
            'transmission': d.get('transmission', 'N/A'), 'fuel': d.get('fuel', 'N/A'), 'color': d.get('color', 'N/A'),
            'body_color': d.get('body_color', 'N/A'), 'inside_color': d.get('inside_color', 'N/A'),
            'body_status': d.get('body_status', 'N/A'), 'body_type': d.get('body_type_fa', 'N/A'),
            'cylinder': d.get('cylinder_fa', 'N/A'), 'volume': s.get('volume', 'N/A'), 'engine': s.get('engine', 'N/A'),
            'acceleration': s.get('acceleration', 'N/A'), 'fuel_consumption': s.get('fuel', 'N/A'),
            'image': d.get('image', 'N/A'), 'image_count': d.get('image_count', 0),
            'brand': d.get('brand', 'N/A'), 'brand_fa': d.get('brand_fa', 'N/A')
        }
    except Exception as e:
        return {'error': str(e), 'ad_code': ad_data.get('detail', {}).get('code', 'unknown')}
        

In [3]:
def save_excel(df, filename):
    msg = "Initial scraping data" if "bama_samand_data.xlsx" in filename else "Complete data"
    try:
        df.to_excel(filename, index=False, engine='openpyxl')
        print(f"💾 {msg} saved to '{filename}'")
    except ImportError:
        print("⚠️ openpyxl not installed. Installing...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "openpyxl"])
        df.to_excel(filename, index=False, engine='openpyxl')
        print(f"💾 {msg} saved to '{filename}'")
    except Exception as e:
        print(f"⚠️ Could not save to Excel: {e}")

def scrape_bama_website_threaded(target_ads=50):
    base_url, all_ads, seen_codes = "https://bama.ir/cad/api/search", [], set()
    params_base = {'yearFrom': '1385-2006', 'yearTo': '', 'vehicle': 'samand'}
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36', 'Accept': 'application/json'}
    
    def fetch_page(page):
        try:
            r = requests.get(base_url, params={**params_base, 'pageIndex': page}, headers=headers, timeout=10)
            data = r.json()
            if not data.get('status'): return []
            ads, m = data.get('data', {}).get('ads', []), data.get('metadata', {})
            print(f"Page {page}: Found {len(ads)} ads (Total: {m.get('total_count', 'unknown')}, Pages: {m.get('total_pages', 'unknown')})")
            page_ads = []
            for ad in ads:
                ad_data = extract_ad_info_from_json(ad)
                if 'error' not in ad_data and (ad_code := ad_data.get('ad_code')) and ad_code not in seen_codes:
                    seen_codes.add(ad_code)
                    page_ads.append(ad_data)
            return page_ads
        except Exception as e:
            print(f"Error on page {page}: {e}")
            return []
    
    print("🚀 Starting API scraping...")
    try:
        r = requests.get(base_url, params={**params_base, 'pageIndex': 1}, headers=headers, timeout=10)
        data = r.json()
        if not data.get('status'): return []
        m = data.get('metadata', {})
        total_pages, total_count = m.get('total_pages', 10), m.get('total_count', 0)
        print(f"📊 Total ads available: {total_count}, Total pages: {total_pages}")
        ads = data.get('data', {}).get('ads', [])
        print(f"Page 1: Found {len(ads)} ads")
        first_page_ads = []
        for ad in ads:
            ad_data = extract_ad_info_from_json(ad)
            if 'error' not in ad_data and (ad_code := ad_data.get('ad_code')) and ad_code not in seen_codes:
                seen_codes.add(ad_code)
                first_page_ads.append(ad_data)
        all_ads.extend(first_page_ads)
        if not first_page_ads: return []
    except Exception as e:
        print(f"⚠️ Error fetching first page: {e}")
        return []
    
    pages_needed = min(total_pages, (target_ads // (len(first_page_ads) or 15)) + 2, total_pages)
    print(f"🎯 Target: {target_ads} ads, will fetch up to {pages_needed} pages")
    
    if len(all_ads) < target_ads and pages_needed > 1:
        pages_to_fetch = list(range(2, min(pages_needed + 1, total_pages + 1)))
        print(f"Fetching pages: {pages_to_fetch}")
        with concurrent.futures.ThreadPoolExecutor(max_workers=5) as executor:
            futures = {executor.submit(fetch_page, p): p for p in pages_to_fetch}
            for f in concurrent.futures.as_completed(futures):
                if page_ads := f.result():
                    all_ads.extend(page_ads)
                    print(f"Progress: {len(all_ads)}/{target_ads} ads collected (from page {futures[f]})")
                    if len(all_ads) >= target_ads:
                        print(f"✅ Reached target of {target_ads} ads, stopping...")
                        [ff.cancel() for ff in futures]
                        break
                time.sleep(0.2)
    
    all_ads = all_ads[:target_ads] if len(all_ads) > target_ads else all_ads
    print(f"✅ Scraping completed! Got {len(all_ads)} unique ads")
    return all_ads

print("📋 STEP 1: Scraping car listings...")
ads_data = scrape_bama_website_threaded(target_ads=50)
df = pd.DataFrame(ads_data)
print(f"✅ Main scraping completed! Found {len(df)} ads\n📊 DataFrame shape: {df.shape}\n📋 Columns: {df.columns.tolist()}\nFirst 5 ads:")
print(df.head())
if not df.empty: print(); save_excel(df, 'bama_samand_data.xlsx')

def scrape_car_details(ad_code, car_url):
    try:
        soup = BeautifulSoup(requests.get(car_url, headers={'User-Agent': 'Mozilla/5.0'}, timeout=15).text, 'html.parser')
        details = {'ad_code': ad_code, 'url': car_url, 'scraped_successfully': True}
        if pe := soup.find('p', class_='price__main'): details['detailed_price'] = pe.text.strip()
        for sel in ['div.ad-page__attributes', 'div.attributes', 'ul.ad-features', 'div.ad-page__item-kv']:
            if sc := soup.select_one(sel):
                for item in sc.find_all(['li', 'div']):
                    if ':' in (t := item.text.strip()): details[t.split(':', 1)[0].strip()] = t.split(':', 1)[1].strip()
        for sel in ['div.ad-page__description', 'div.description', 'div.ad-page__description-text']:
            if de := soup.select_one(sel):
                details['description'] = de.text.strip()[:300]
                break
        print(f"✅ Scraped details for: {ad_code}")
        return details
    except Exception as e:
        print(f"❌ Failed to scrape {ad_code}: {e}")
        return {'ad_code': ad_code, 'url': car_url, 'scraped_successfully': False, 'error': str(e)}

def scrape_all_car_details(df, max_workers=2):
    print(f"🚗 Starting detailed scraping for {len(df)} cars...")
    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = [executor.submit(scrape_car_details, row.get('ad_code', 'unknown'), row['link']) for _, row in df.iterrows() if 'link' in row and row['link'] != 'N/A']
        results = []
        for i, f in enumerate(concurrent.futures.as_completed(futures), 1):
            results.append(f.result())
            if i % 5 == 0: print(f"Progress: {i}/{len(futures)}")
        return results

valid_links = df[df['link'].notna() & (df['link'] != 'N/A')].copy()
print(f"\n🔗 STEP 2: Valid links to scrape: {len(valid_links)}")
print("🎯 STEP 3: Scraping detailed information (first 10 cars)...")
details_df = pd.DataFrame(scrape_all_car_details(valid_links.head(10), max_workers=2))
print(f"📊 Detailed data shape: {details_df.shape}")

print("\n🔗 STEP 4: Merging data...")
if not details_df.empty:
    merged_df = pd.merge(df, details_df, on='ad_code', how='left', suffixes=('_basic', '_detailed'))
    print(f"📊 Merged DataFrame shape: {merged_df.shape}\n✅ Successfully scraped: {merged_df['scraped_successfully'].sum()} cars\n🎯 FINAL RESULTS:\nAvailable columns: {merged_df.columns.tolist()}\nColumns with data: {sum(merged_df[col].notna().any() for col in merged_df.columns)}")
    sample_cols = ['ad_code', 'price_basic', 'location_basic', 'year_basic', 'detailed_price', 'description']
    available = [c for c in sample_cols if c in merged_df.columns]
    print(f"\n📋 Sample data:")
    print(merged_df[available].head(10).to_string(index=False) if available else merged_df.head(10).to_string(index=False))
    merged_df.to_csv('bama_samand_complete_data.csv', index=False, encoding='utf-8-sig')
    print("\n💾 Complete data saved to 'bama_samand_complete_data.csv'")
    save_excel(merged_df, 'bama_samand_complete_data.xlsx')
    final_df = merged_df
else:
    print("❌ No detailed data was scraped")
    final_df = df
print("\n✅ ALL DONE! You can now use 'final_df' for analysis")

📋 STEP 1: Scraping car listings...
🚀 Starting API scraping...
📊 Total ads available: 41, Total pages: 3
Page 1: Found 30 ads
🎯 Target: 50 ads, will fetch up to 3 pages
Fetching pages: [2, 3]
Page 2: Found 30 ads (Total: 71, Pages: 4)
Progress: 60/50 ads collected (from page 2)
✅ Reached target of 50 ads, stopping...
Page 3: Found 30 ads (Total: 101, Pages: 5)
✅ Scraping completed! Got 50 unique ads
✅ Main scraping completed! Found 50 ads
📊 DataFrame shape: (50, 27)
📋 Columns: ['ad_code', 'title', 'subtitle', 'link', 'price', 'price_type', 'location', 'time_posted', 'year', 'mileage', 'trim', 'transmission', 'fuel', 'color', 'body_color', 'inside_color', 'body_status', 'body_type', 'cylinder', 'volume', 'engine', 'acceleration', 'fuel_consumption', 'image', 'image_count', 'brand', 'brand_fa']
First 5 ads:
    ad_code       title                 subtitle  \
0  oqjeairs    سمند، LX               1385 | XU7   
1  2tubbohx    سمند، LX               1394 | EF7   
2  cvwihfxt  سمند، سورن  140

In [4]:
final_df

,ad_code,title,subtitle,link,price,price_type,location,time_posted,year,mileage,...,engine,acceleration,fuel_consumption,image,image_count,brand,brand_fa,url,scraped_successfully,description
0,oqjeairs,سمند، LX,1385 | XU7,https://bama.ir/car/detail-oqjeairs-samand-lx-...,"355,000,000",lumpsum,پاکدشت,9 ساعت پیش,1385,"240,000 km",...,4 سیلندر XU7,13.1 ثانیه,8.3 لیتر در صد کیلومتر,https://cdn-sth1.bama.ir/uploads/BamaImages/Ve...,3,samand,سمند,https://bama.ir/car/detail-oqjeairs-samand-lx-...,True,NaN
1,2tubbohx,سمند، LX,1394 | EF7,https://bama.ir/car/detail-2tubbohx-samand-lx-...,"480,000,000",lumpsum,اسلام شهر,10 ساعت پیش,1394,"230,000 km",...,4 سیلندر EF7,12 ثانیه,7.5 لیتر در صد کیلومتر,https://cdn-sth1.bama.ir/uploads/BamaImages/Ve...,3,samand,سمند,https://bama.ir/car/detail-2tubbohx-samand-lx-...,True,توضیحات دیسک و صفحه نو\r\nگیربکس کامل سرویس شد...
2,cvwihfxt,سمند، سورن,1404 | پلاس XU7P بنزینی,https://bama.ir/car/detail-cvwihfxt-samand-sor...,"880,000,000",lumpsum,سمنان,11 ساعت پیش,1404,صفر کیلومتر,...,4 سیلندر XU7P,13 ثانیه,7 لیتر در صد کیلومتر,None,0,samand,سمند,https://bama.ir/car/detail-cvwihfxt-samand-sor...,True,NaN
3,1srb7g5i,سمند، LX,1396 | EF7,https://bama.ir/car/detail-1srb7g5i-samand-lx-...,"568,000,000",lumpsum,کرج / 45 متری گلشهر,11 ساعت پیش,1396,"249,000 km",...,4 سیلندر EF7,12 ثانیه,7.5 لیتر در صد کیلومتر,https://cdn-sth1.bama.ir/uploads/BamaImages/Ve...,3,samand,سمند,https://bama.ir/car/detail-1srb7g5i-samand-lx-...,True,توضیحات خودرو سالم\r\nکف صندوق خوردگی\r\n✔️خود...
4,svk7lnw5,سمند، سورن,1404 | پلاس XU7P بنزینی,https://bama.ir/car/detail-svk7lnw5-samand-sor...,"885,000,000",lumpsum,اسلام شهر,11 ساعت پیش,1404,صفر کیلومتر,...,4 سیلندر XU7P,13 ثانیه,7 لیتر در صد کیلومتر,https://cdn-sth1.bama.ir/uploads/BamaImages/Ve...,4,samand,سمند,https://bama.ir/car/detail-svk7lnw5-samand-sor...,True,توضیحات دریچه برقی با کارت طلایی
5,azhr3kmf,سمند، سورن,1404 | پلاس XU7P بنزینی,https://bama.ir/car/detail-azhr3kmf-samand-sor...,"938,000,000",lumpsum,کرج / حسین‌آباد,11 ساعت پیش,1404,صفر کیلومتر,...,4 سیلندر XU7P,13 ثانیه,7 لیتر در صد کیلومتر,None,0,samand,سمند,https://bama.ir/car/detail-azhr3kmf-samand-sor...,True,NaN
6,mquvxlkn,سمند، سورن,1404 | پلاس XU7P بنزینی,https://bama.ir/car/detail-mquvxlkn-samand-sor...,0,negotiable,سقز,11 ساعت پیش,1404,صفر کیلومتر,...,4 سیلندر XU7P,13 ثانیه,7 لیتر در صد کیلومتر,None,0,samand,سمند,https://bama.ir/car/detail-mquvxlkn-samand-sor...,True,توضیحات تحویلی ۲۷ آبان۴۰۴ ،نقد
7,jzspw7vr,سمند، سورن,1388 | ساده,https://bama.ir/car/detail-jzspw7vr-samand-sor...,"450,000,000",lumpsum,زاهدان,12 ساعت پیش,1388,"214,000 km",...,None,None,None,https://cdn-sth1.bama.ir/uploads/BamaImages/Ve...,4,samand,سمند,https://bama.ir/car/detail-jzspw7vr-samand-sor...,True,توضیحات واقعی، لاستیک ۱۰۰ درصد، بیمه ۱۲ ماه، ب...
8,xodyqh9i,سمند، سورن,1404 | پلاس XU7P بنزینی,https://bama.ir/car/detail-xodyqh9i-samand-sor...,"950,000,000",lumpsum,دزفول,12 ساعت پیش,1404,صفر کیلومتر,...,4 سیلندر XU7P,13 ثانیه,7 لیتر در صد کیلومتر,https://cdn-sth1.bama.ir/uploads/BamaImages/Ve...,2,samand,سمند,https://bama.ir/car/detail-xodyqh9i-samand-sor...,True,توضیحات تحویل از نمایندگی ۳ روز پیش\r\n دریچه...
9,2qtnob6z,سمند، سورن,1403 | پلاس EF7 دوگانه سوز,https://bama.ir/car/detail-2qtnob6z-samand-sor...,"1,000,000,000",lumpsum,کرمانشاه,12 ساعت پیش,1403,"23,000 km",...,4 سیلندر EF7 دوگانه سوز,12.8 ثانیه,7.5 لیتر در صد کیلومتر,https://cdn-sth1.bama.ir/uploads/BamaImages/Ve...,4,samand,سمند,https://bama.ir/car/detail-2qtnob6z-samand-sor...,True,توضیحات درحد صفر. رینگ اسپورت.لاستیک خارجی ۲۰۵...
